# Using `matcher.py`

This notebook shows how to import and use `AdaptiveTemplateMatching` from the packaged module in `src/adaptive_template_matching/matcher.py`.

Before running the examples, install the package from the repository root:

```bash
pip install -e git+https://github.com/Garangatang/adaptive-template-matching-repo/tree/main
```

## 1. Import the matcher

The package exposes `AdaptiveTemplateMatching` at the top level.

In [ ]:
from adaptive_template_matching import AdaptiveTemplateMatching
import numpy as np

## 2. Import example data from .pkl file

Upsamp_UP_Dict.pkl contains underfoot pressure data from six participants utilized in the paper associated with this repo.

In [ ]:
def openPkl(file_path):
    with open(f"{file_path}.pkl", 'rb') as f:
        return_data = pkl.load(f)

    return return_data

def saveToPkl(file_path, data):
    with open(f"{file_path}.pkl", "wb") as f:
        pkl.dump(data, f)

In [ ]:
dataDict = openPkl("data/Upsamp_UP_Dict.pkl")

## 3. Create a matcher instance

You can use the defaults or customize the template geometry.

In [ ]:
# Thresholds/gates
AMP_MAX = 0.30
THR_PASS = 0.80
REL_ERR = 0.40
PASSES = 2

# Sum absolute difference controls
POS_SHIFT = 30
TEMPLATE_LEN = 200
NSAD_THRESH     = int(-0.3 * _template_len)
MIN_STD        = 1e-3

# Low-amplitude gating
ENFORCE_LOW_AMP = True
LOW_AMP_QUANTILE = 0.35
LOW_AMP_COVER     = 0.50

# clustering
CLUSTER_BY = "sad"   # {"r", "sad", "sad_shifted"}
CLUSTER_RADIUS = 1000    # at 1980 Hz, set per expected inter-event spacing

# Diagnostics
SHOW_DEBUG     = False
DEBUG_VERBOS  = False

# Template construction parameters
CHANGE_POINT_ARR = [125, 180]
ANGLE_ARR = [80, 85]
BASELINE = 0
TEMPLATE_SCALER = 0.12
REFLECT = False

matcher = AdaptiveTemplateMatching(
    cp_inds=CHANGE_POINT_ARR,
    template_angls=ANGLE_ARR,
    template_len=TEMPLATE_LEN,
    baseline=BASELINE,
    template_scaler=TEMPLATE_SCALER,
    reflect=REFLECT,
)

## 4. Run a scan

Tune the thresholds to match your signal scaling and data quality.

Available options for run functions are: cold_start_run_dataset, warm_start_run_dataset, and all_data_run_dataset.

In [ ]:
indices, scores = matcher.scan_matches(
    data=signal,
    threshold=0.6,
    amp_max=1.0,
    rel_err=0.5,
    flat_err_thresh=0.2,
    pos_shift=20,
    sad_thresh=0.0,
    min_std=1e-3,
    enforce_low_amp=False,
    show_debug=False,
)

indices, scores[:10] if len(scores) else scores

## 5. Update the template

If you want the template to adapt to accepted segments, call `update_template`.

In [ ]:
matcher.update_template(signal, np.asarray(indices), show_debug=False)
matcher.get_template_shape()[:10]

## 6. Run dataset helpers

For multi-signal workflows, pass a dictionary of dataset names to signal arrays.

In [ ]:
data_dict = {
    "trial_1": signal,
    "trial_2": signal + 0.002 * np.random.randn(len(signal)),
}

results = matcher.cold_start_run_dataset(
    data_dict=data_dict,
    amp_max=1.0,
    thr=0.6,
    rel_err=0.5,
    passes=1,
    pos_shift=20,
    sad_thresh=0.0,
    min_std=1e-3,
    enforce_low_amp=False,
    show_debug=False,
    debug_verbose=False,
)

results

## Notes

- `scan_matches` is the lowest-level matching call.
- `final_template_match_and_plot` is useful when you want summary output and optional plots.
- `cold_start_run_dataset`, `warm_start_run_dataset`, and `all_data_run_dataset` are convenient for multi-dataset adaptation workflows.